In [ ]:
# | default_exp sureau_soil_params

In [ ]:
# | hide

from fastcore import *
from pprint import pprint
from nbdev.showdoc import *

In [ ]:
# | export
import numpy as np
from plant_hydraulics.utils import convert_FtoV
from plant_hydraulics.parameter_classes import SurEauSoilParams, SurEauVegetationParams

In [ ]:
# | export


def compute_soil_root_geometry(
    soil: SurEauSoilParams,
    veg: SurEauVegetationParams,
) -> SurEauSoilParams:
    """Compute soil parameters depending on plant parameters (Gardner-Cowan geometry).
    Old function name init_par_soil
    """
    b = 1 / np.sqrt(np.pi * veg.Lv)
    soil.B_GC = veg.La * 2 * np.pi / np.log(b / veg.root_radius)

    return soil

In [ ]:
# | export


def sureau_soil_params(soil: SurEauSoilParams) -> SurEauSoilParams:
    """
    Compute derived soil hydraulic quantities from raw SurEau soil parameters.

    Broadcasts scalar inputs to per-layer arrays, computes layer thicknesses,
    van Genuchten or Campbell derived parameters, and converts volumetric
    water contents to water heights (mm) for field capacity and wilting point.

    __Parameters:__

    - soil: SurEauSoilParams object with the following input attributes:

        - depth: Cumulative depth of each soil layer bottom (m).
        - RFC: Rock fragment content for each layer (%).
        - g_soil0: Maximum soil surface conductance to water vapor (mmol/m2/s).
        - offset_psoil: Offset applied to soil water potential for each layer (MPa).
        - psoil_at_field_capacity: Soil water potential at field capacity (MPa).
        - reset_SWC: Whether to reset soil water content to field capacity each year (bool).
        - water_soil_transfer: Whether inter-layer water transfer is enabled (bool).
        - soil_evap: Whether soil surface evaporation is enabled (bool).
        - PTF: Pedotransfer function to use: "VG" (van Genuchten) or "Campbell".
        - Ksat: Saturated hydraulic conductivity for each layer (mm/h).
        - saturation_capacity: Volumetric water content at saturation for each layer (m3/m3).
        - residual_capacity: Residual volumetric water content for each layer (m3/m3).
        - alpha_vg: Van Genuchten alpha parameter for each layer (cm-1).
        - n_vg: Van Genuchten n shape parameter for each layer (-).
        - I_vg: Van Genuchten tortuosity parameter for each layer (-).
        - b_camp: Campbell b shape parameter for each layer (-).
        - psie_camp: Campbell air-entry potential for each layer (MPa).

    __Returns:__

    - soil: Updated SurEauSoilParams object with the following derived attributes:

        - n_layers: Number of soil layers (-).
        - layer_thickness: Thickness of each soil layer (m).
        - m: Van Genuchten m parameter, computed as 1 - 1/n_vg (-).
        - V_field_capacity: Water height at field capacity for each layer (mm).
        - V_saturation_capacity: Water height at saturation for each layer (mm).
        - V_residual_capacity: Water height at residual content for each layer (mm).
        - V_wilting_point: Water height at wilting point (-1.5 MPa) for each layer (mm).
        - B_GC: Gardner-Cowan geometry factor for each layer (-). Set to 0.5 placeholder if not previously computed from root geometry.

    Parameters
    ----------
    soil : SurEauSoilParams
        SurEau soil parameters object with the following input attributes:
        - depth : np.ndarray
            Cumulative depth of each soil layer bottom (m).
        - RFC : np.ndarray
            Rock fragment content for each layer (%).
        - g_soil0 : float
            Maximum soil surface conductance to water vapor (mmol/m2/s).
        - offset_psoil : np.ndarray
            Offset applied to soil water potential for each layer (MPa).
        - psoil_at_field_capacity : float
            Soil water potential at field capacity (MPa).
        - reset_SWC : bool
            Whether to reset soil water content to field capacity each year.
        - water_soil_transfer : bool
            Whether inter-layer water transfer is enabled.
        - soil_evap : bool
            Whether soil surface evaporation is enabled.
        - PTF : str
            Pedotransfer function to use: "VG" (van Genuchten) or "Campbell".
        - Ksat : np.ndarray
            Saturated hydraulic conductivity for each layer (mm/h).
        - saturation_capacity : np.ndarray
            Volumetric water content at saturation for each layer (m3/m3).
        - residual_capacity : np.ndarray
            Residual volumetric water content for each layer (m3/m3).
        - alpha_vg : np.ndarray
            Van Genuchten alpha parameter for each layer (cm-1).
        - n_vg : np.ndarray
            Van Genuchten n shape parameter for each layer (-).
        - I_vg : np.ndarray
            Van Genuchten tortuosity parameter for each layer (-).
        - b_camp : np.ndarray
            Campbell b shape parameter for each layer (-).
        - psie_camp : np.ndarray
            Campbell air-entry potential for each layer (MPa).

    Returns
    -------
    soil : SurEauSoilParams
        Updated SurEau soil parameters object with the following derived attributes:
        - n_layers : int
            Number of soil layers (-).
        - layer_thickness : np.ndarray
            Thickness of each soil layer (m).
        - m : np.ndarray
            Van Genuchten m parameter, computed as 1 - 1/n_vg (-).
        - V_field_capacity : np.ndarray
            Water height at field capacity for each layer (mm).
        - V_saturation_capacity : np.ndarray
            Water height at saturation for each layer (mm).
        - V_residual_capacity : np.ndarray
            Water height at residual content for each layer (mm).
        - V_wilting_point : np.ndarray
            Water height at wilting point (-1.5 MPa) for each layer (mm).
        - B_GC : np.ndarray
            Gardner-Cowan geometry factor for each layer (-).
            Set to 0.5 placeholder if not previously computed from root geometry.
    """

    depth = np.asarray(soil.depth)
    n = len(depth)
    soil.n_layers = n

    # Ensure arrays are properly sized ------------------------------------------
    for attr in [
        "RFC",
        "offset_psoil",
        "Ksat",
        "saturation_capacity",
        "residual_capacity",
        "alpha_vg",
        "n_vg",
        "I_vg",
        "b_camp",
        "psie_camp",
    ]:
        val = np.asarray(getattr(soil, attr))
        if val.ndim == 0 or len(val) == 1:
            setattr(soil, attr, np.full(n, float(val.flat[0])))

    # Layer thickness -----------------------------------------------------------
    soil.layer_thickness = np.concatenate([[depth[0]], np.diff(depth)])

    # VG m derived parameter-----------------------------------------------------
    soil.m = 1 - 1 / soil.n_vg

    # Volumetric capacities -----------------------------------------------------
    # Same as θ_sat in equation 8.8 but here express as water stock. For example
    # a soil layer is 30 cm thick with θ_sat = 0.4 soil.V_saturation_capacity 
    # will have 12 cm of water (0.4 × 30 cm) 
    soil.V_saturation_capacity = convert_FtoV(
        soil.saturation_capacity, soil.RFC, soil.layer_thickness
    )
    
    # Same as θ_res in equation 8.8 but here express as water stock.
    soil.V_residual_capacity = convert_FtoV(
        soil.residual_capacity, soil.RFC, soil.layer_thickness
    )

    # Field capacity from psoil_at_field_capacity -------------------------------
    if soil.PTF == "VG":
        theta_fc = (
            soil.residual_capacity
            + (soil.saturation_capacity - soil.residual_capacity)
            / (1 + (soil.alpha_vg * soil.psoil_at_field_capacity * 10000) ** soil.n_vg)
            ** soil.m
        )
    else:
        theta_fc = soil.saturation_capacity * (
            soil.psoil_at_field_capacity / soil.psie_camp
        ) ** (-1 / soil.b_camp)
    soil.V_field_capacity = convert_FtoV(theta_fc, soil.RFC, soil.layer_thickness)

    # Wilting point at -1.5 MPa -------------------------------------------------
    if soil.PTF == "VG":
        theta_wp = (
            soil.residual_capacity
            + (soil.saturation_capacity - soil.residual_capacity)
            / (1 + (soil.alpha_vg * 1.5 * 10000) ** soil.n_vg) ** soil.m
        )
    else:
        theta_wp = soil.saturation_capacity * (1.5 / soil.psie_camp) ** (
            -1 / soil.b_camp
        )
    soil.V_wilting_point = convert_FtoV(theta_wp, soil.RFC, soil.layer_thickness)

    # Temporary B_GC. Overwritten by compute_soil_root_geometry() once
    # vegetation parameters (La, Lv) are available
    if soil.B_GC is None:
        soil.B_GC = np.ones(soil.n_layers) * 0.5

    return soil

### Example sureau_soil_params()

In [ ]:
from plant_hydraulics.parameter_classes import SurEauSoilParams, SurEauSoil
from plant_hydraulics.sureau_soil_params import sureau_soil_params

In [ ]:
soil_params = SurEauSoilParams(
    depth=np.array([0.2, 0.8, 2.0]),
    RFC=np.array([75.0, 75.0, 75.0]),
    g_soil0=30.0,
    PTF="VG",
    Ksat=np.array([1.69, 1.69, 1.69]),
    saturation_capacity=np.array([0.50, 0.50, 0.50]),
    residual_capacity=np.array([0.098, 0.098, 0.098]),
    alpha_vg=np.array([0.0005, 0.0005, 0.0005]),
    n_vg=np.array([1.55, 1.55, 1.55]),
    I_vg=np.array([0.5, 0.5, 0.5]),
)

In [ ]:
soil_params = sureau_soil_params(soil_params)